## Ranking model: predicting which stocks should be ranked highest each day

In this section, I build a ranking model that orders stocks from the most promising to the least promising in terms of **next-day return**.

I use a **pointwise ranking approach**:
- first, train one generic model to predict `Target_Return_1d`
- then, for each date, sort stocks by the predicted return in descending order

I chose this approach for three reasons:
1. it directly matches the project goal of ranking stocks by possible daily gains
2. it keeps the modeling framework simple and interpretable
3. it uses the same generic model across all stocks, which is consistent with the rest of the project

A pairwise or listwise learning-to-rank method could also be used, but for a small universe of stocks and an academic project, predicting returns first and ranking second is a strong and defendable choice.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from pathlib import Path

pd.set_option("display.max_columns", None)

In [2]:
data_path = Path("../01_data/raw/df_eda.csv")
df_model = pd.read_csv(data_path)

print("Shape:", df_model.shape)
df_model.head()

Shape: (19836, 17)


,Date,Open,High,Low,Close,Adj Close,Volume,Ticker,Return_1d,Return_3d,Return_5d,Volatility_5d,Volatility_20d,Volume_Change_1d,Target_Return_1d,Weekday,Month
0,2010-01-04,7.622500,7.660714,7.585000,7.643214,6.412384,493729600,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,0.001729,Monday,January
1,2010-01-05,7.664286,7.699643,7.616071,7.656429,6.423470,601904800,AAPL,0.001729,NaN,NaN,NaN,NaN,0.219098,-0.015906,Tuesday,January
2,2010-01-06,7.656429,7.686786,7.526786,7.534643,6.321297,552160000,AAPL,-0.015906,NaN,NaN,NaN,NaN,-0.082646,-0.001849,Wednesday,January
3,2010-01-07,7.562500,7.571429,7.466071,7.520714,6.309608,477131200,AAPL,-0.001849,-0.016028,NaN,NaN,NaN,-0.135882,0.006648,Thursday,January
4,2010-01-08,7.510714,7.571429,7.466429,7.570714,6.351556,447610800,AAPL,0.006648,-0.011195,NaN,NaN,NaN,-0.061871,-0.008821,Friday,January


## Prepare the ranking dataset

The ranking target is still the next-day return, because the final ranking should reflect which stock is expected to have the highest daily gain.

I keep the dataset in long format, where each row is one stock on one date.  
This is convenient because the model can learn from all stocks together, and later I can rank them **within each date**.

In [5]:
ranking_df = df_model.copy()

ranking_df["Date"] = pd.to_datetime(ranking_df["Date"])
ranking_df = ranking_df.sort_values(["Date", "Ticker"]).reset_index(drop=True)

target_col = "Target_Return_1d"

candidate_numeric_features = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Return_1d",
    "Return_3d",
    "Return_5d",
    "Volatility_5d",
    "Volatility_20d",
    "Volume_Change_1d"
]

candidate_categorical_features = ["Ticker", "Weekday", "Month"]

numeric_features = [col for col in candidate_numeric_features if col in ranking_df.columns]
categorical_features = [col for col in candidate_categorical_features if col in ranking_df.columns]

required_cols = ["Date", "Ticker", target_col] + numeric_features + categorical_features
ranking_df = ranking_df[required_cols].copy()

# We need the target to evaluate ranking quality
ranking_df = ranking_df.dropna(subset=[target_col]).copy()

# Keep only dates with at least 2 stocks available, otherwise ranking is meaningless
valid_dates = (
    ranking_df.groupby("Date")["Ticker"]
    .nunique()
    .loc[lambda s: s >= 2]
    .index
)

ranking_df = ranking_df[ranking_df["Date"].isin(valid_dates)].copy()

ranking_df["Actual_Rank"] = (
    ranking_df.groupby("Date")[target_col]
    .rank(method="first", ascending=False)
)

print("Shape:", ranking_df.shape)
print("Number of ranking dates:", ranking_df["Date"].nunique())
ranking_df.head()

ValueError: Cannot index with multidimensional key

## Time-based train, validation, and test split

Because this is financial time-series data, I split by time rather than randomly.  
This avoids leakage from the future into the past and makes the ranking task closer to a real forecasting setup.

I use:
- training set for fitting the models
- validation set for model selection
- test set for final evaluation

In [ ]:
unique_dates = np.array(sorted(ranking_df["Date"].unique()))

n_dates = len(unique_dates)
train_cut = int(n_dates * 0.70)
val_cut = int(n_dates * 0.85)

train_dates = unique_dates[:train_cut]
val_dates = unique_dates[train_cut:val_cut]
test_dates = unique_dates[val_cut:]

train_df = ranking_df[ranking_df["Date"].isin(train_dates)].copy()
val_df = ranking_df[ranking_df["Date"].isin(val_dates)].copy()
test_df = ranking_df[ranking_df["Date"].isin(test_dates)].copy()

for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(
        f"{name}: {part['Date'].min().date()} -> {part['Date'].max().date()} | "
        f"rows={len(part)} | dates={part['Date'].nunique()}"
    )

## Define ranking metrics

A ranking model should not be judged only by regression error.  
What matters is whether the model places the best opportunities near the top.

I use three ranking-oriented metrics:
- **NDCG@3**: checks whether the best stocks are placed near the top 3 positions
- **Spearman correlation**: measures how well the predicted order matches the true order
- **Top-1 hit rate**: shows how often the model correctly identifies the single best stock of the day

I also track the realized return of the top-ranked stock to see whether the chosen top idea is better than the average stock on that day.

In [ ]:
def dcg_at_k(relevance, k):
    relevance = np.asarray(relevance)[:k]
    if len(relevance) == 0:
        return np.nan
    discounts = np.log2(np.arange(2, len(relevance) + 2))
    return np.sum((2**relevance - 1) / discounts)


def evaluate_ranking(df, score_col, target_col="Target_Return_1d", k=3):
    daily_rows = []

    for date, g in df.groupby("Date"):
        g = g.dropna(subset=[score_col, target_col]).copy()

        if len(g) < 2:
            continue

        g = g.sort_values(score_col, ascending=False).reset_index(drop=True)
        n = len(g)

        # Convert actual ranks into positive relevance scores for NDCG
        actual_rank = g[target_col].rank(method="first", ascending=False)
        relevance = (n - actual_rank + 1).astype(int).to_numpy()

        dcg = dcg_at_k(relevance, min(k, n))
        idcg = dcg_at_k(np.sort(relevance)[::-1], min(k, n))
        ndcg = dcg / idcg if idcg > 0 else np.nan

        spearman = g[[score_col, target_col]].corr(method="spearman").iloc[0, 1]

        predicted_top = g.iloc[0]["Ticker"]
        actual_top = g.loc[g[target_col].idxmax(), "Ticker"]
        top1_hit = int(predicted_top == actual_top)

        daily_rows.append({
            "Date": date,
            "NDCG@3": ndcg,
            "Spearman": spearman,
            "Top1_Hit": top1_hit,
            "Predicted_Top1": predicted_top,
            "Actual_Top1": actual_top,
            "Top1_Actual_Return": g.iloc[0][target_col],
            "Best_Actual_Return": g[target_col].max(),
            "Universe_Mean_Return": g[target_col].mean()
        })

    daily_eval = pd.DataFrame(daily_rows)

    summary = pd.DataFrame({
        "Metric": ["NDCG@3", "Spearman", "Top1_Hit", "Top1_Actual_Return", "Best_Actual_Return", "Universe_Mean_Return"],
        "Value": [
            daily_eval["NDCG@3"].mean(),
            daily_eval["Spearman"].mean(),
            daily_eval["Top1_Hit"].mean(),
            daily_eval["Top1_Actual_Return"].mean(),
            daily_eval["Best_Actual_Return"].mean(),
            daily_eval["Universe_Mean_Return"].mean()
        ]
    })

    return daily_eval, summary

## Build a naive ranking baseline

Before using machine learning, I create a simple benchmark:
rank stocks by today's `Return_1d`.

This is a momentum-style baseline.  
It is useful because a more complex model should beat a very simple ranking rule to justify its use.

In [ ]:
baseline_score_col = "Return_1d"

val_baseline_daily, val_baseline_summary = evaluate_ranking(
    val_df,
    score_col=baseline_score_col,
    target_col=target_col
)

test_baseline_daily, test_baseline_summary = evaluate_ranking(
    test_df,
    score_col=baseline_score_col,
    target_col=target_col
)

print("Validation baseline:")
display(val_baseline_summary)

print("Test baseline:")
display(test_baseline_summary)

## Compare a few candidate models on the validation set

I compare three pointwise models:
- **Linear Regression**: a simple linear benchmark
- **Random Forest**: captures nonlinear patterns and interactions
- **LightGBM**: usually strong on structured tabular data and efficient on mixed signals

I keep the same feature set and preprocessing for all models so the comparison is fair.

I choose the final model based on validation ranking quality, not just on regression logic.

In [ ]:
feature_cols = numeric_features + categorical_features

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

candidate_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ),
    "LightGBM": LGBMRegressor(
        objective="regression",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
}

validation_results = []

for model_name, model in candidate_models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(train_df[feature_cols], train_df[target_col])

    val_scored = val_df.copy()
    val_scored["Predicted_Score"] = pipe.predict(val_df[feature_cols])

    _, val_summary = evaluate_ranking(
        val_scored,
        score_col="Predicted_Score",
        target_col=target_col
    )

    row = val_summary.set_index("Metric")["Value"].to_dict()
    row["Model"] = model_name
    validation_results.append(row)

validation_comparison = (
    pd.DataFrame(validation_results)
    .set_index("Model")
    .sort_values("NDCG@3", ascending=False)
)

validation_comparison